<a href="https://colab.research.google.com/github/Limsungrae/Hongong/blob/main/%ED%98%BC%EA%B3%B5%EB%A8%B8%EC%8B%A05_3%ED%8A%B8%EB%A6%AC%EC%9D%98_%EC%95%99%EC%83%81%EB%B8%94.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 트리의 앙상블
- 여러 개의 결정트리를 만들어서 하나처럼 사용하는 방법

In [2]:
## 데이터 준비
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
wine = pd.read_csv('https://bit.ly/wine_csv_data')
data = wine[['alcohol', 'sugar', 'pH']]
target = wine['class']
# 훈련(80%) / 테스트(20%) 데이터 분할
train_input, test_input, train_target, test_target = train_test_split(
    data, target, test_size=0.2, random_state=42)

## 랜덤 포레스트 (RandomForest)
- 여러 개의 결정트리를 만들어 평균냄
- 과적합 줄이는 대표 모델

In [3]:
from sklearn.model_selection import cross_validate  # 교차검증
from sklearn.ensemble import RandomForestClassifier  # 랜덤포레스트 모델

# 랜덤포레스트 생성 (n_jobs-1 = ,모든 CPU 사용)
rf = RandomForestClassifier(n_jobs=-1, random_state=42)

# 교차검증 수행 (훈련 점수도 같이 반환)
scores = cross_validate(rf, train_input, train_target,
                        return_train_score=True, n_jobs=-1)

# 평균 훈련 정확도 / 검증 정확도 출력
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

# 과적합 발생

0.9973541965122431 0.8905151032797809


In [4]:
# 전체 훈련 데이터로 모델 학습
rf.fit(train_input, train_target)

# 각 특성의 중요도 출력
print(rf.feature_importances_)
#feature_importances_ = 어떤 변수(alcohol, sugar, pH)가 중요한지 알려줌 (중요도)
# 중요도는 정보량이 아닌 선택된 횟수 기반

[0.23167441 0.50039841 0.26792718]


In [6]:
# OOB (Out Of Bag) 점수 사용
# oob = 랜덤포레스트에서 자동 검증 기능
rf = RandomForestClassifier(oob_score=True, n_jobs=-1, random_state=42)

# 학습
rf.fit(train_input, train_target)

# OOB 점수 출력 (검증세트 없이 평가 가능)
print(rf.oob_score_)

0.8934000384837406


## 엑스트라 트리 (Extra Trees)

In [7]:
from sklearn.ensemble import ExtraTreesClassifier  # 엑스트라트리

# 모델 생성
et = ExtraTreesClassifier(n_jobs=-1, random_state=42)

# 교차검증 수행
scores = cross_validate(et, train_input, train_target,
                        return_train_score=True, n_jobs=-1)

# 평균 정확도 출력
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9974503966084433 0.8887848893166506


## 랜덤포레스트 vs 엑스트라트리

- 랜덤포레스트 = 최적 분할 찾음
- 엑스트라트리 = 랜덤으로 분할 (더 빠름, 약간 성능 낮을 수 있음)

In [9]:
# 학습
et.fit(train_input, train_target)

# 특성 중요도 출력
print(et.feature_importances_)

[0.20183568 0.52242907 0.27573525]


In [10]:
# w중요도 이름과 연결하기
feature_names = ['alcohol','sugar','pH']
for name,score in zip(feature_names,et.feature_importances_):
    print(name,score)

alcohol 0.20183568183615255
sugar 0.5224290729926174
pH 0.27573524517123


In [22]:
indices = np.argsort(-et.feature_importances_)
for i in indices:
    print(feature_names[i], et.feature_importances_[i])

sugar 0.5224290729926174
pH 0.27573524517123
alcohol 0.20183568183615255


## 그레이디언트 부스팅
- 이전 모델의 오차를 계속 보완하면서 학습

In [23]:
from sklearn.ensemble import GradientBoostingClassifier  # 그레이디언트 부스팅

gb = GradientBoostingClassifier(random_state=42)

# 교차검증 수행
scores = cross_validate(gb, train_input, train_target,
                        return_train_score=True, n_jobs=-1)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.8881086892152563 0.8720430147331015


## 핵심 파라미터

- n_estimators → 트리 개수
- learning_rate → 학습 속도

In [ ]:
# 트리 개수 증가 + 학습률 증가
gb = GradientBoostingClassifier(n_estimators=500, learning_rate=0.2,
                                random_state=42)

# 교차검증
scores = cross_validate(gb, train_input, train_target,
                        return_train_score=True, n_jobs=-1)

# 성능 출력
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

In [24]:
# 학습
gb.fit(train_input, train_target)

# 특성 중요도 출력
print(gb.feature_importances_)

[0.11949941 0.74871836 0.13178224]


## 히스토그램 기반 부스팅 (HistGradientBoosting)
- 속도 빠름 (대용량 데이터에 강함)

In [25]:
from sklearn.ensemble import HistGradientBoostingClassifier

# 모델 생성
hgb = HistGradientBoostingClassifier(random_state=42)

# 교차검증
scores = cross_validate(hgb, train_input, train_target,
                        return_train_score=True, n_jobs=-1)

# 성능 출력
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9321723946453317 0.8801241948619236


In [27]:
from sklearn.inspection import permutation_importance  # 중요도 평가

# 학습
hgb.fit(train_input, train_target)

# 중요도 계산 (훈련 데이터 기준)
# permutation_importance = 값을 섞어서 성능이 얼마나 떨어지는지 확인(진짜중요도)
result = permutation_importance(hgb, train_input, train_target,n_repeats=10,
random_state=42, n_jobs=-1)
#n_repeats=10, =  10번 반복후 편균 구하기

# 평균 중요도 출력
print(result.importances_mean)

[0.08876275 0.23438522 0.08027708]


In [28]:
# 테스트 데이터 기준 중요도
result = permutation_importance(
    hgb, test_input, test_target,
    n_repeats=10, random_state=42, n_jobs=-1)

print(result.importances_mean)

[0.05969231 0.20238462 0.049     ]


In [29]:
# 테스트 정확도 출력 (최종 성능)
hgb.score(test_input, test_target)

0.8723076923076923

## XGBoost
- 부스팅 계열 끝판왕
- 대회에서 많이 사용

In [ ]:
from xgboost import XGBClassifier  # XGBoost 모델

# 모델 생성 (hist 방식)
xgb = XGBClassifier(tree_method='hist', random_state=42)

# sklearn 호환 설정
xgb._estimator_type = "classifier"

# 교차검증
scores = cross_validate(xgb, train_input, train_target,
                        return_train_score=True, n_jobs=-1)

# 성능 출력
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

## LightGBM
- XGBoost보다 빠름
- 대용량 데이터에 강함

In [30]:
from lightgbm import LGBMClassifier  # LightGBM 모델

# 모델 생성
lgb = LGBMClassifier(random_state=42)

# 교차검증
scores = cross_validate(lgb, train_input, train_target,
                        return_train_score=True, n_jobs=-1)

# 성능 출력
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.935828414851749 0.8801251203079884


| 모델       | 특징             |
| -------- | -------------- |
| 랜덤포레스트   | 안정적, 과적합 줄임    |
| 엑스트라트리   | 더 랜덤, 더 빠름     |
| 그라디언트부스팅 | 성능 좋지만 느림      |
| 히스토그램GB  | 빠르고 효율적        |
| XGBoost  | 성능 최고 수준       |
| LightGBM | 속도 + 성능 둘 다 좋음 |
